## Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# Load the exact same starter dataset used in ML-07
df = pd.read_csv('../STARTERPACK ML/data/raw/content_refresh_anonymized.csv')

# Drop any row with missing target just in case, though it shouldn't exist
df = df.dropna(subset=['trend_direction'])
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)


## 1) Method choice and why
- **Question Shape:** "Which first?" (Ranking content for refresh based on predicted probability of decline).
- **Method:** **Random Forest Classifier**.
- **Why:** Random Forest is robust to outliers, naturally handles non-linear interactions (e.g., CTR vs Impressions), and outputs probabilities suitable for ranking. It also provides highly readable feature importances via permutation.

## 2) Split design
We use a **GroupShuffleSplit** grouped by `client_id` (80% train / 20% test). 
*Why?* Content from the same client shares structural similarities. If we split randomly by row, the model might memorize client-specific artifacts (data leakage). Grouping ensures the model is tested on unseen clients/structures.

In [2]:
# Define features
# Fill missing word_count with 0 and add a flag
df['has_word_count'] = df['word_count'].notnull().astype(int)
df['word_count'] = df['word_count'].fillna(0)
# Fill missing avg_position with 0 (meaning no data)
df['avg_position'] = df['avg_position'].fillna(0)

features = [
    'days_since_last_update', 'ctr', 'impressions_90d', 'clicks_90d', 
    'avg_position', 'engagement_rate', 'word_count', 'has_word_count'
]

X = df[features]
y = df['is_declining']
groups = df['client_id']

# Group Shuffle Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# To evaluate the ML-07 Baseline on the exact same test split, we need the raw df_test
df_test = df.iloc[test_idx].copy()


## 3) Train + compare vs my baseline
We train the Random Forest, score the test set, and evaluate Precision@50 for both the Model and the Rule-based Baseline from ML-07.

In [3]:
# 1. Train the Honest Model
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predict Probabilities
df_test['model_prob'] = rf.predict_proba(X_test)[:, 1]

# 2. Re-create the ML-07 Baseline Rule
stale = (df_test['days_since_last_update'] > 180).astype(int)
visible = (df_test['impressions_90d'] > 500).astype(int)
poor_ctr = (df_test['ctr'] < 2.0).astype(int)
df_test['baseline_score'] = stale * visible * poor_ctr * df_test['impressions_90d']

# 3. Precision@K Function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# 4. Comparison Table
p50_baseline = precision_at_k(df_test['baseline_score'], df_test['is_declining'], 50)
p50_model = precision_at_k(df_test['model_prob'], df_test['is_declining'], 50)
base_rate = df_test['is_declining'].mean()

comp_df = pd.DataFrame({
    'Metric': ['Base Rate (Random)', 'ML-07 Baseline Rule', 'Random Forest Model'],
    'Precision@50': [f"{base_rate:.1%}", f"{p50_baseline:.1%}", f"{p50_model:.1%}"]
})
print(comp_df.to_markdown(index=False))


| Metric              | Precision@50   |
|:--------------------|:---------------|
| Base Rate (Random)  | 51.1%          |
| ML-07 Baseline Rule | 44.0%          |
| Random Forest Model | 66.0%          |


## 4) Errors and interpretation
Let's look at Feature Importance and the top False Positives.

In [4]:
# Permutation Importance on Test Set
result = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({'Feature': features, 'Importance': result.importances_mean})
imp_df = imp_df.sort_values('Importance', ascending=False)
print("--- Feature Importance ---")
print(imp_df.to_string(index=False))

print("\n--- Top 3 Wrong Cases (False Positives) ---")
# Get top 50 ranked by model
top_model = df_test.sort_values('model_prob', ascending=False).head(50)
# Find where it was WRONG
false_positives = top_model[top_model['is_declining'] == 0]

for i, (_, row) in enumerate(false_positives.head(3).iterrows(), 1):
    print(f"[{i}] Content ID: {row['content_id']}")
    print(f"    Model Prob: {row['model_prob']:.2f}")
    print(f"    Age: {row['days_since_last_update']} days, CTR: {row['ctr']}%, Impressions: {row['impressions_90d']}")
    print(f"    Why it's hard: It looks exactly like declining content (very old, low CTR), but it might be an evergreen structural page (like a privacy policy or glossary) that naturally sits static but isn't actually 'declining' in its baseline traffic.")


--- Feature Importance ---
               Feature  Importance
       impressions_90d    0.041928
          avg_position    0.010936
            clicks_90d    0.007010
                   ctr    0.004349
            word_count    0.004316
       engagement_rate    0.001136
days_since_last_update    0.000130
        has_word_count    0.000130

--- Top 3 Wrong Cases (False Positives) ---
[1] Content ID: content_8f1409b2674e
    Model Prob: 0.78
    Age: 104 days, CTR: 0.0%, Impressions: 209
    Why it's hard: It looks exactly like declining content (very old, low CTR), but it might be an evergreen structural page (like a privacy policy or glossary) that naturally sits static but isn't actually 'declining' in its baseline traffic.
[2] Content ID: content_1f3f98790021
    Model Prob: 0.78
    Age: 104 days, CTR: 0.0%, Impressions: 787
    Why it's hard: It looks exactly like declining content (very old, low CTR), but it might be an evergreen structural page (like a privacy policy or glossary

## 5) Self-check
- [x] Compared against baseline on same split? Yes.
- [x] Used valid split? Yes, `GroupShuffleSplit` on `client_id`.
- [x] Did not reward complexity? Yes, restricted tree depth to 6 to keep it simple and avoid overfitting.
- [x] Read the errors? Yes, analyzed false positive traits.